# Notebook 00 -- extract_openends

**What this notebook asks:** turn the raw Qualtrics export into one row per answer.

Every participant read one of three vignettes, set a sentence, and then answered
three open-ended questions about why. This notebook reads the raw Qualtrics
**Values** export and writes `openends_long.csv` -- one row per `response_id` x
`item_type`, three rows per participant, with the vignette-split sentence items
already coalesced (gathered from the three parallel columns into one).

| column | meaning |
|---|---|
| `response_id` | `ResponseId`, the participant key every later file joins on |
| `vignette` | the embedded `Vignette` field (1 / 2 / 3) |
| `item_type` | `ranking` / `backward` / `forward` |
| `orientation` | `backward` for ranking + backward, `forward` for forward |
| `text` | the verbatim open-end |
| `n_words` | `[A-Za-z']+` token count -- the prereg 6.7 verbosity covariate |

Column mapping. **The three items are scored separately and are never
concatenated** (prereg 5.15):

| `item_type` | export column | note |
|---|---|---|
| `ranking` | `Priorit_Explain` | this is the prereg's **Q333**; not vignette-split |
| `backward` | `Sentence_OpenEnd1_V{1,2,3}` | **coalesced by the `Vignette` field** |
| `forward` | `Sentence_OpenEnd2_V{1,2,3}` | **coalesced by the `Vignette` field** |

Use the **Values** export, not Labels. Labels is a codebook only: it writes anchor
wording for the endpoints of the 7-point scales and bare numbers for the middle
points, so it can never be used numerically. The two exports are identical for
open-ends, sliders, sentences and `Vignette`, so the text would be the same either
way -- but one rule for which file to touch keeps the numeric trap out of every
notebook downstream.

**Sample.** The raw export holds 1,015 responses. The preregistered exclusions
(finished, then the two attention checks: 1015 -> 965 -> 958 -> **948**, prereg 6.2)
are applied in Track 1's `01_main_analysis.Rmd` and reach this pipeline through
`retribution_scored.csv`. This notebook restricts the corpus to those 948 analytic
participants, so the file it writes should hold **948 x 3 = 2,844 rows**.

This is the one manual step in the whole chain. Every notebook after this one reads
files the chain itself wrote.

## Config

In [1]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "00"                     # this notebook's number
NB_NAME     = "extract_openends"       # this notebook's job, in two words
NB_ASKS     = "Turn the raw Qualtrics export into one row per answer."

## Setup

In [2]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

NOTEBOOK 00 -- extract_openends
  Turn the raw Qualtrics export into one row per answer.
  mode: RUN_MODE=full, HUMAN_GOLD=False
  folder: /content/retribution_nlp
  this folder is empty: notebook 00 is the place to start.


## Load

Two inputs, and one upload dialog between them.

- **The raw Qualtrics Values export** (`FullData_Values.csv`, the complete collection
  of 1,015 responses). `Data/` also holds an interim partial export (`Pilot_300_*`);
  always read the full file. If a copy is already sitting in the working folder it is
  used as it stands, so re-running this notebook costs no second upload.
- **`retribution_scored.csv`**, the scored survey file Track 1's `01_main_analysis.Rmd`
  writes. It is optional here, and it is what defines the analytic sample of 948: put
  it into the same dialog. Without it the corpus stays at all 1,015 raw participants
  and this notebook says so, loudly, in its output.

In [3]:
# === IMPORTS ===
# WHAT THIS CELL DOES: loads the few libraries this notebook uses and sets display
# options so the printed checks stay legible.

# The whole notebook is DataFrame work: the raw export in, the long open-ends out.
import pandas as pd
# re     -> the n_words tokenizer, the one regular expression (a text-matching
#           pattern) in this notebook.
# shutil -> copying an uploaded file to its canonical name.
import re, shutil
# Counter tallies how many participants resolved a source column per item. That tally
# is the evidence the extraction check reads, and without it a renamed export column
# would empty a whole item in silence.
from collections import Counter

# files.upload() is how the raw export enters the chain. There is deliberately no
# drive.mount anywhere in this pipeline: the notebooks hand CSVs to each other through
# the working folder, so nothing depends on one person's Drive layout, every input is
# an explicit named file, and the notebooks run unchanged on any account. Outside
# Colab there is no browser to talk to, so the import is allowed to fail and the
# notebook falls back to whatever is already in the working folder.
try:
    from google.colab import files
    IN_COLAB = True
except Exception as _e:
    files, IN_COLAB = None, False
    print(f"  (not running in Colab: {_e})")

import warnings
# Silences pandas chatter that would otherwise bury the checks this notebook prints.
# HAZARD: this also hides warnings worth seeing. Every condition that actually matters
# here is therefore an explicit assert or an explicit print, never a warning.
# ONE CATEGORY, NOT EVERY WARNING. A bare filterwarnings('ignore') stays in
# force for the rest of the runtime and hides warnings that are real signals,
# not noise. Name the category so only the intended class is quieted.
warnings.filterwarnings('ignore', category=FutureWarning)

# Wide enough that the per-item diagnostic tables stay on one line.
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print("[ok] imports complete")

[ok] imports complete


In [5]:
# === THE ONE MANUAL STEP: FIND THE RAW QUALTRICS VALUES EXPORT ===
# WHAT THIS CELL DOES: locates the raw export, and the optional survey scaffold,
# either in the working folder (the re-run path) or through a single upload dialog.
# WHY THIS IS THE ONLY UPLOAD IN THE CHAIN: every notebook after this one reads files
# the pipeline itself wrote, so this is the single place where a hand-picked file can
# be the wrong file. That is also why the Values-versus-Labels check lives here and
# nowhere else.

RAW_DEFAULT    = 'FullData_Values.csv'      # the name the raw export usually carries
SURVEY_DEFAULT = 'retribution_scored.csv'   # Track 1's scored survey file, N = 948

# The re-run path first. A file already in the working folder is used as it stands.
raw_path    = need(RAW_DEFAULT, optional=True)
survey_path = need(SURVEY_DEFAULT, optional=True)

if raw_path is None:
    # An export copied in under a different name is still an export, so the folder is
    # searched before the browser is bothered. This also keeps a Labels export that is
    # sitting in the folder visible to the check in the next cell, instead of letting
    # it be ignored and then re-uploaded by hand. Values-named files sort first.
    _here = [f for f in sorted(os.listdir(WORKDIR)) if f.lower().endswith('.csv')
             and any(k in f.lower() for k in ('values', 'labels', 'fulldata'))]
    _here.sort(key=lambda f: 0 if 'values' in f.lower() else 1)
    if _here:
        raw_path = os.path.join(WORKDIR, _here[0])
        print(f"  [ok]   {_here[0]}  (found in the working folder)")

uploaded = {}
if raw_path is None or survey_path is None:
    print("\nUpload the RAW Qualtrics VALUES export -- FullData_Values.csv, the full")
    print("1,015-response collection, NOT the Pilot_300 interim file.")
    print("Add retribution_scored.csv to the same dialog if you have it: that file is")
    print("what restricts this corpus to the 948 analytic participants.")
    if IN_COLAB:
        try:
            uploaded = files.upload()
        except Exception as _e:          # dialog cancelled, or no browser attached
            print(f"  (upload skipped: {_e})")
    else:
        print(f"  (no upload dialog outside Colab: copy the files into {WORKDIR} "
              "and re-run this cell)")

def _pick(substr):
    """First uploaded file whose name contains substr, or None.

    Colab prefixes a re-uploaded file with '(1)', '(2)', and so on, so this matches on
    a piece of the name rather than on an exact filename. Otherwise a second upload in
    the same session silently fails to be found.
    """
    hits = [f for f in uploaded if substr in f]
    return hits[0] if hits else None

if raw_path is None:
    # Prefer a filename containing "Values" when several files were dropped in; fall
    # back to whatever arrived that is not the survey scaffold.
    _cands = ([f for f in uploaded if 'Values' in f]
              or [f for f in uploaded if 'retribution_scored' not in f])
    raw_path = _cands[0] if _cands else None

if raw_path is None:
    # A missing raw export is the one thing this notebook genuinely cannot work
    # around, so it stops here with a sentence rather than failing further down.
    raise MissingInput(
        "\n\n  MISSING INPUT: the raw Qualtrics Values export.\n"
        "  Re-run this cell and choose FullData_Values.csv in the dialog, or copy it\n"
        f"  into {WORKDIR} under that name and re-run.\n")

  [ok]   FullData_Values.csv  (written unknown time)
  [skip] retribution_scored.csv is not here. This notebook will carry on without it and say so in its output.

Upload the RAW Qualtrics VALUES export -- FullData_Values.csv, the full
1,015-response collection, NOT the Pilot_300 interim file.
Add retribution_scored.csv to the same dialog if you have it: that file is
what restricts this corpus to the 948 analytic participants.


Saving retribution_scored.csv to retribution_scored.csv


In [6]:
# === VALUES OR LABELS: the one file-choice trap in this study ===
if 'labels' in os.path.basename(raw_path).lower():
    # The two exports are identical for open-ends, sliders, sentences and Vignette, so
    # the TEXT would be fine either way -- but the Labels export writes anchor wording
    # for the endpoints of the 7-point scales and bare numbers for the middle points,
    # so it can never be used numerically. Refusing it here keeps one rule ("the
    # pipeline reads Values") instead of a rule with an exception, which is what keeps
    # the numeric trap out of the notebooks that DO read scale columns.
    raise ValueError(
        f'{raw_path} is the LABELS export. Labels is a CODEBOOK ONLY. '
        'Re-run the cell above with the ..._Values.csv export.')

# The survey scaffold, if one arrived in the dialog. It is copied to its canonical
# name so the notebooks after this one can ask for one filename even when Colab handed
# us "retribution_scored (1).csv".
if survey_path is None:
    _s = _pick('retribution_scored')
    if _s is not None:
        if os.path.basename(_s) != SURVEY_DEFAULT:
            shutil.copyfile(_s, os.path.join(WORKDIR, SURVEY_DEFAULT))
        survey_path = os.path.join(WORKDIR, SURVEY_DEFAULT)
        print(f"[ok]   {SURVEY_DEFAULT} taken from the upload.")

print(f"\nReading raw export : {raw_path}")
print(f"Analytic sample    : {survey_path if survey_path else 'not supplied (see below)'}")

[ok]   retribution_scored.csv taken from the upload.

Reading raw export : /content/retribution_nlp/FullData_Values.csv
Analytic sample    : /content/retribution_nlp/retribution_scored.csv


## Do

Read the export, split each participant into three rows, check that no item lost its
source column, restrict to the analytic sample, and count everything out loud.

In [7]:
# === THE THREE ITEMS, AND WHERE EACH ONE LIVES IN THE EXPORT ===
# WHY LONG AND NOT WIDE: prereg 5.15 scores the three items SEPARATELY and never as
# one combined passage. The backward-looking justification and the ranking explanation
# are where desert and proportionality language lives; the forward-looking
# accomplishment item is where deterrence, incapacitation and rehabilitation language
# lives; and concatenating them would mix exactly the two registers the study is
# trying to keep apart. Long shape makes that structural: every scorer in the chain
# (the word list, the model ensemble, the embeddings) runs per row, and item_type
# travels with every score. Never rebuild a text_combined column
# (MEASUREMENT_DESIGN.md 3).

# item_type -> (orientation, export column stem, vignette-split?)
# orientation collapses the three items onto the backward/forward contrast that E6
# measures: the ranking explanation and the sentence justification both look back at
# what the offender did, the accomplishment item looks ahead. Note that orientation is
# a coarser label than item_type and never replaces it -- E3 is tested on Q333 and Q1
# SEPARATELY, never pooled (MEASUREMENT_DESIGN.md 3), so item_type is the key.
# The third element is the vignette-split flag: the two sentence open-ends were asked
# inside the randomly assigned vignette block and therefore land in three parallel
# columns, one per vignette; the ranking explanation was asked once, outside the
# randomizer, so it arrives in a single column.
ITEMS = {
    'ranking':  ('backward', 'Priorit_Explain',   False),  # prereg tag Q333
    'backward': ('backward', 'Sentence_OpenEnd1', True),
    'forward':  ('forward',  'Sentence_OpenEnd2', True),
}

# The three codes the vignette randomizer assigns. A participant who left the survey
# before the vignette block carries a BLANK Vignette field: they were never shown
# either sentence open-end, so there is no _V1/_V2/_V3 column to read for them, and
# their backward and forward rows are empty by construction. That count is the
# denominator the extraction check holds those two items to.
VALID_VIGNETTES = {'1', '2', '3'}


def _word_count(t):
    """Verbosity covariate (prereg 6.7): count of [A-Za-z']+ tokens."""
    # The apostrophe is inside the character class so "don't" counts as one word
    # rather than two. Digits and punctuation are not tokens, so "20 years" counts as
    # one word. `t or ''` absorbs None and NaN-as-empty without raising.
    # n_words matters downstream because the item medians differ sharply (32 words for
    # ranking, 19 backward, 13 forward): word count is a covariate in every model, and
    # the dictionary emits per-100-word rates as well as raw counts, so a longer
    # answer cannot look more retributive merely by being longer.
    return len(re.findall(r"[A-Za-z']+", t or ''))

In [8]:
# === READ THE RAW EXPORT ===
def _read_values_export(raw_csv_path):
    """Qualtrics Values export -> a DataFrame with one row per participant.

    Qualtrics ships three header rows: (0) column names, (1) question text,
    (2) the ImportId JSON. Rows 1-2 are dropped, row 0 is kept as the header.
    """
    # dtype=str keeps every field as text: ResponseId and Vignette must not be coerced
    # to numbers, and an open-end that happens to read "20" must stay a string.
    # encoding='utf-8-sig' strips the byte-order mark Qualtrics writes, which would
    # otherwise ride along on the first column name. keep_default_na=False is the
    # important one: a participant who skipped an open-end comes through as '' rather
    # than as a missing value, so .strip() never raises and a skipped answer is scored
    # as a blank (n_words = 0) instead of quietly disappearing.
    raw = pd.read_csv(raw_csv_path, skiprows=[1, 2], dtype=str,
                      encoding='utf-8-sig', keep_default_na=False)
    # Qualtrics occasionally pads header names; strip so the lookups below match.
    raw.columns = [str(c).strip() for c in raw.columns]

    # Two structural checks, both fatal on purpose. Without ResponseId nothing can be
    # joined to the survey scaffold; without Vignette the split sentence items cannot
    # be coalesced and would silently come back empty for every participant.
    assert 'ResponseId' in raw.columns, 'Not a Qualtrics export: no ResponseId column.'
    assert 'Vignette' in raw.columns, \
        'No Vignette column -- the split sentence open-ends cannot be coalesced.'
    return raw

In [9]:
# === ONE ROW PER PARTICIPANT IN, THREE ROWS PER PARTICIPANT OUT ===
def _extract_long_rows(raw):
    """Build the long frame, and tally where each item's text came from.

    Returns the long frame and, beside it, `resolved` -- a count, per item, of how
    many participants got a real source column. That count is the evidence the
    extraction check in the next cell reads.
    """
    rows, resolved = [], Counter()
    for _, r in raw.iterrows():
        # The embedded field recording which of the three vignettes this participant
        # saw (prereg 3.2). It is the key to the coalescing.
        vig = str(r.get('Vignette', '')).strip()
        for item, (orient, stem, split) in ITEMS.items():
            if split:
                if vig in VALID_VIGNETTES:
                    # Pick the column for the vignette THIS participant saw. Everyone
                    # who reached the block answered exactly one of _V1/_V2/_V3 and
                    # left the other two blank, so reading the wrong one returns an
                    # empty string rather than an error -- which is precisely why the
                    # resolution is driven off the Vignette field and then checked.
                    col = f'{stem}_V{vig}'
                    if col not in raw.columns:
                        # a future export may already ship a coalesced column
                        col = stem if stem in raw.columns else None
                else:
                    # No vignette, so neither sentence open-end was ever asked of this
                    # participant. The row is still written -- with empty text and
                    # n_words = 0 -- because that is the truthful record: an item that
                    # does not exist for this person, not a lookup that failed. These
                    # rows carry no source column and are counted OUT of the check's
                    # denominator rather than counted as failures.
                    col = None
            else:
                # Not vignette-split: Priorit_Explain sits outside the randomizer, so
                # it resolves for every participant, blank Vignette included.
                col = stem if stem in raw.columns else None
            # A column that does not exist yields '' here; the tally below is what
            # turns a genuine resolution failure into a loud one.
            text = str(r.get(col, '')).strip() if col else ''
            if col:
                resolved[item] += 1
            rows.append({
                'response_id': str(r.get('ResponseId', '')).strip(),
                'vignette':    vig,
                'item_type':   item,
                'orientation': orient,
                'text':        text,
                'n_words':     _word_count(text),
            })
    return pd.DataFrame(rows), resolved

In [10]:
# === EXTRACTION CHECK: a silently unresolved column would empty a whole item ===
def _check_extraction(out, raw, resolved):
    """Stop if an item lost its source column, or if the shape is not three-per-person."""
    # This is the guard the design calls for by name: the prereg tags the ranking
    # explanation Q333 while the export calls it Priorit_Explain, and a rename on
    # either side would otherwise drop a whole item without a single error message,
    # leaving E3 to run on two items instead of three (MEASUREMENT_DESIGN.md 3).
    n_resp = len(raw)
    n_vig = int(raw['Vignette'].astype(str).str.strip().isin(VALID_VIGNETTES).sum())
    for item, (_orient, _stem, split) in ITEMS.items():
        # Each item is held to ITS OWN denominator. The ranking explanation was asked
        # outside the randomizer, so all n_resp participants must resolve a column for
        # it. The two vignette-split items exist only for the participants who reached
        # the vignette block, so that is what they are checked against. The check stays
        # a real tripwire on a renamed or dropped column while treating an early exit
        # as the legitimate thing it is.
        expected = n_vig if split else n_resp
        assert resolved[item] == expected, (
            f'{item}: resolved a source column for only {resolved[item]}/{expected} '
            f'participants -- check the {ITEMS[item][1]} / Vignette columns.')
    # Long shape contract: exactly three rows per participant, no more, no fewer.
    assert len(out) == n_resp * 3, f'Expected {n_resp * 3} long rows, got {len(out)}'
    # A duplicated or blank ResponseId would break every downstream merge: the survey
    # scaffold join, the SEED/TEST split assignment, every per-facet score file.
    assert out.response_id.nunique() == n_resp, 'Duplicate or blank ResponseId values'
    return n_resp, n_vig

In [11]:
# === RUN THE EXTRACTION ===
raw = _read_values_export(raw_path)
openends, resolved = _extract_long_rows(raw)
n_resp, n_vig = _check_extraction(openends, raw, resolved)

# The resolution log. Printing where each item came from is what makes the
# Q333/Priorit_Explain mismatch visible in the notebook's own output, not only in the
# check that would have fired had it gone wrong.
print('[Q333 resolution] item_type=="ranking" was extracted from "Priorit_Explain".')
print('[coalescing]      Sentence_OpenEnd1/_2 taken from the _V1/_V2/_V3 column')
print("                  matching each participant's Vignette field.")

# Named in the output rather than left to be rediscovered from a non-blank count that
# looks short: these participants have no vignette, so the two sentence items are
# empty for them by construction.
_no_vig = int(openends.loc[~openends.vignette.isin(VALID_VIGNETTES),
                           'response_id'].nunique())
if _no_vig:
    print(f'[no vignette]     {_no_vig} participants left before the vignette block and')
    print('                  were never shown either sentence open-end, so their')
    print('                  backward and forward rows are empty. Their ranking row')
    print('                  still resolves -- Priorit_Explain sits outside the block.')

print(f'\nExtracted {len(openends)} rows from {n_resp} raw responses '
      f'({n_resp} participants x 3 items); {n_vig} reached the vignette block.')

[Q333 resolution] item_type=="ranking" was extracted from "Priorit_Explain".
[coalescing]      Sentence_OpenEnd1/_2 taken from the _V1/_V2/_V3 column
                  matching each participant's Vignette field.
[no vignette]     38 participants left before the vignette block and
                  were never shown either sentence open-end, so their
                  backward and forward rows are empty. Their ranking row
                  still resolves -- Priorit_Explain sits outside the block.

Extracted 3045 rows from 1015 raw responses (1015 participants x 3 items); 977 reached the vignette block.


In [12]:
# === RESTRICT TO THE ANALYTIC SAMPLE (prereg 6.2: 1015 -> 965 -> 958 -> 948) ===
# retribution_scored.csv IS the analytic sample: Track 1's 01_main_analysis.Rmd
# applies the preregistered exclusions (finished, then the two attention checks)
# before writing it.
#
# WHY THE RESTRICTION HAPPENS HERE, AT THE SOURCE. Everything downstream is computed
# on the file this notebook writes: the word-list coverage floors, the inter-model
# agreement (ICC, the intraclass correlation that gate clause B reads), the
# facet-versus-survey correlations of gate clause C. Carrying the 67 excluded
# participants forward would put them inside every one of those numbers, and clipping
# the sample afterwards on the R side would not undo it. It also spends model calls on
# responses no analysis will ever read.
if survey_path is not None:
    _surv = pd.read_csv(survey_path)
    # The survey export names the participant key ResponseId; some hand-edited copies
    # carry it already renamed. Accept either so the join key is one column here.
    _idcol = next((c for c in ('ResponseId', 'response_id', 'ResponseID')
                   if c in _surv.columns), None)
    if _idcol is None:
        raise ValueError(
            'No id column found in ' + os.path.basename(survey_path) + '. Looked for '
            'ResponseId, response_id and ResponseID. Columns present: '
            + ', '.join(map(str, _surv.columns[:12])))
    _keep = set(_surv[_idcol].astype(str))
    _before = openends.response_id.nunique()
    openends = openends[openends.response_id.astype(str).isin(_keep)].reset_index(drop=True)
    _after = openends.response_id.nunique()
    # TRIPWIRE. retribution_scored.csv is produced by the R Markdown, not by this
    # chain, so there is nothing guaranteeing its id column lines up with the
    # Qualtrics export. If it does not, the filter above throws away every row and
    # the corpus is silently empty. Every notebook after this one would then run on
    # nothing, and the first error would point at the item_type column, which is not
    # where the problem is. Stop here, where the cause is still visible.
    if _after == 0:
        raise ValueError(
            'The analytic restriction matched NOTHING: every participant was dropped. '
            'That means the two id columns do not line up, not that everyone was '
            'excluded. Check that the ' + _idcol + ' values in '
            + os.path.basename(survey_path) + ' look like the response_id values in '
            'the Qualtrics export.')
    if _after < 0.5 * len(_keep):
        print(f'  WARNING: only {_after} of the {len(_keep)} participants in the survey '
              'file were matched. A near-complete match is expected; a shortfall this '
              'large means the two id columns disagree.')
    ANALYTIC_RESTRICTION = 'applied'
    print(f'Analytic restriction applied: {_before} -> '
          f'{openends.response_id.nunique()} participants ({len(openends)} rows), '
          f'against the {len(_keep)} in {os.path.basename(survey_path)}.')
else:
    ANALYTIC_RESTRICTION = 'not run'
    print('[not run] Analytic restriction: retribution_scored.csv is not here, so the')
    print('          exclusions are not knowable in this notebook. The corpus keeps all')
    print(f'          {openends.response_id.nunique()} raw participants, the 67 that '
          'prereg 6.2 excludes included.')
    print('          Every agreement and correlation number downstream would then be')
    print('          computed on a sample the study does not report. Add the file and')
    print('          re-run this notebook before any scoring.')

Analytic restriction applied: 1015 -> 948 participants (2844 rows), against the 948 in retribution_scored.csv.


In [13]:
# === SMOKE MODE: the first 20 participants, whole ===
# RUN_MODE = "smoke" exists to prove the chain runs end to end for pennies. The trim
# takes whole PARTICIPANTS rather than the first 20 rows, because every check in this
# pipeline is written against three rows per participant: a trim that cut a person in
# half would fail those checks for a reason that has nothing to do with the data.
if RUN_MODE == 'smoke':
    _ids = openends.response_id.drop_duplicates().head(20)
    openends = openends[openends.response_id.isin(_ids)].reset_index(drop=True)
    print(f'SMOKE MODE: kept the first {len(_ids)} participants, {len(openends)} rows. '
          'Set RUN_MODE = "full" for the real corpus.')
else:
    print('RUN_MODE = "full": the whole corpus goes forward.')

RUN_MODE = "full": the whole corpus goes forward.


In [14]:
# === VALIDATE LOUDLY, PART 1: the shape contract the whole chain depends on ===
# WHAT BREAKS WITHOUT THESE CHECKS: none of these failures raises anywhere
# downstream. A dropped item just quietly removes a third of the corpus; a collapsed
# item silently makes every later score a score of the wrong text. They have to be
# caught here, where the file is made.
N_PART = openends.response_id.nunique()
N_ROWS = len(openends)
TARGET = 948 * 3      # the analytic sample: 948 participants x 3 open-ended items

# issubset rather than equality, so an export carrying extra columns still passes.
expected_cols = {'response_id', 'vignette', 'item_type', 'orientation', 'text', 'n_words'}
assert expected_cols.issubset(set(openends.columns)), \
    f'Missing columns: {expected_cols - set(openends.columns)}'

# Exact set equality here, deliberately: a missing item_type means a whole open-end
# went missing, and an extra one means something was concatenated or renamed. Either
# would silently change what E3 and E6 are computed on.
assert set(openends.item_type.unique()) == {'ranking', 'backward', 'forward'}, \
    f'Unexpected item_type values: {sorted(openends.item_type.unique())}'

# ONE ROW PER PERSON PER ITEM. This is the check that proves the three answers were
# not collapsed into one passage: if any (participant, item) pair carried more than
# one row the three items would have been merged, duplicated or fanned out by a bad
# join, and the "scored separately, never concatenated" rule of prereg 5.15 would be
# broken in the very first file of the chain.
_per_cell = openends.groupby(['response_id', 'item_type']).size()
assert int(_per_cell.max()) == 1, (
    'More than one row for some participant x item_type -- the three open-ends are '
    'not one-per-cell any more.')
assert N_ROWS == N_PART * 3, \
    f'Expected {N_PART * 3} rows for {N_PART} participants, got {N_ROWS}'

print(f'Rows: {N_ROWS} = {N_PART} participants x 3 items, one row per person per item.')
if RUN_MODE == 'smoke':
    print(f'  Smoke mode, so the {TARGET}-row target does not apply.')
elif N_ROWS == TARGET:
    print(f'  [ok] This is the expected count: 948 analytic participants x 3 = {TARGET}.')
else:
    print(f'  [check] The design expects {TARGET} rows (948 x 3) and this file has '
          f'{N_ROWS}.')
    print(f'          Analytic restriction: {ANALYTIC_RESTRICTION}. An unrestricted '
          'run gives 1015 x 3 = 3045.')

Rows: 2844 = 948 participants x 3 items, one row per person per item.
  [ok] This is the expected count: 948 analytic participants x 3 = 2844.


In [15]:
# === VALIDATE LOUDLY, PART 2: is there anything in the text? ===
# Belt and braces on top of keep_default_na=False upstream: fillna('') so a blank
# open-end is an empty string rather than a missing value (later notebooks lowercase
# and pad `text` directly, and a missing value there would raise), and astype(str)
# covers a column pandas typed as a number when every value was empty.
openends['text'] = openends['text'].fillna('').astype(str)
# n_words was written as a whole number above. Coercing it here keeps a blank cell in
# a hand-edited file reading as 0 instead of turning the whole column into text, which
# every `n_words > 0` filter in the chain depends on.
openends['n_words'] = pd.to_numeric(openends['n_words'],
                                    errors='coerce').fillna(0).astype(int)

# WHAT THESE THREE LINES MEAN: all three items should be non-blank for very nearly
# every participant (the items use a soft prompt, prereg 5.15, so a few blanks are
# expected), and the length ordering should be ranking > backward > forward -- median
# 32 / 19 / 13 words in the first tranche. A collapsed count, or a median of 0 for one
# item, is the signature of a column-resolution failure upstream, not of a quiet
# sample. The mean is printed beside the median because the ranking item is
# right-skewed (median 32, mean about 44).
print('Non-blank counts and word length per item_type:')
for it in ['ranking', 'backward', 'forward']:
    sub = openends[(openends.item_type == it) & (openends.n_words > 0)]
    med = int(sub.n_words.median()) if len(sub) else 0
    avg = sub.n_words.mean() if len(sub) else 0.0
    print(f'  {it:9s}: {len(sub):5d} non-blank of {N_PART:5d}, median {med:3d} words '
          f'(mean {avg:.1f})')

# Blank rows are kept, never dropped. A participant who skipped an item is a real
# record with n_words = 0; deleting the row would make the file uneven and would hide
# the skip from every count that follows.
_blank = int((openends.n_words == 0).sum())
print(f'  blank rows kept in place: {_blank} of {N_ROWS} '
      '(skipped answers stay in the file as empty text, n_words = 0)')
print('\nVignette balance (participants):')
print(openends[openends.item_type == 'ranking']
      .vignette.value_counts().sort_index().to_string())

Non-blank counts and word length per item_type:
  ranking  :   945 non-blank of   948, median  35 words (mean 44.3)
  backward :   947 non-blank of   948, median  20 words (mean 25.8)
  forward  :   945 non-blank of   948, median  14 words (mean 18.2)
  blank rows kept in place: 7 of 2844 (skipped answers stay in the file as empty text, n_words = 0)

Vignette balance (participants):
vignette
1    315
2    317
3    316


## Save and check

In [16]:
# === SAVE ===
# openends_long.csv is the spine of the whole chain: the word list, the model
# ensemble and the embeddings all score these rows, and every later file joins back to
# them on (response_id, item_type).
OUT = 'openends_long.csv'
# THE CONTRACT, ENFORCED ON THE FILE THIS NOTEBOOK WRITES. Every notebook
# downstream keys on (response_id, item_type) and several merge on it. A
# duplicate here silently multiplies rows in notebooks 02, 04, 09, 10 and 11:
# a checker proved it by appending three duplicated rows and watching the E4,
# E6 and E7 sample sizes climb from 948 to 951 while the printed line still
# read 948, with every p-value in the corrected family shrinking. Check it
# here, at the source, rather than asking each consumer to find it later.
assert not openends.duplicated(['response_id', 'item_type']).any(), (
    'openends_long.csv is NOT one row per participant per question. '
    'Duplicated keys: '
    + str(openends[openends.duplicated(['response_id', 'item_type'],
                                       keep=False)][['response_id',
                                                     'item_type']].head(6)))
openends.to_csv(OUT, index=False)
done(OUT, rows=len(openends),
     note=(f'{N_PART} participants x 3 items; analytic restriction '
           f'{ANALYTIC_RESTRICTION}'))

# The survey scaffold is registered too, when it is here, so the notebooks that need
# the closed-ended scales (gate clause C correlates each facet with its matching
# survey scale) can ask for it by name instead of asking for another upload. It was
# written by Track 1, not by this notebook; the note records that.
if survey_path is not None:
    done(SURVEY_DEFAULT, rows=len(pd.read_csv(survey_path)),
         note='written by Track 1 01_main_analysis.Rmd; carried into the chain here')

print(f'\nNOTEBOOK {NB_ID} -- {NB_NAME}: done.')
print(f'  {OUT}: {N_ROWS} rows = {N_PART} participants x 3 items '
      f'(ranking, backward, forward).')
print(f'  Expected in a full run: {TARGET} rows (948 analytic participants x 3).')
print(f'  Analytic restriction (prereg 6.2, applied upstream in Track 1): '
      f'{ANALYTIC_RESTRICTION}.')
print(f'  Run mode: {RUN_MODE}.')
print('\n  NEXT: notebook 01. It reads openends_long.csv from this same folder, so')
print('  there is nothing to upload again -- this was the one manual step.')

  [wrote] openends_long.csv  (2844 rows, 594,111 bytes)
  [wrote] retribution_scored.csv  (948 rows, 1,817,122 bytes)

NOTEBOOK 00 -- extract_openends: done.
  openends_long.csv: 2844 rows = 948 participants x 3 items (ranking, backward, forward).
  Expected in a full run: 2844 rows (948 analytic participants x 3).
  Analytic restriction (prereg 6.2, applied upstream in Track 1): applied.
  Run mode: full.

  NEXT: notebook 01. It reads openends_long.csv from this same folder, so
  there is nothing to upload again -- this was the one manual step.
